In [ ]:
%load_ext autoreload
%autoreload 2

# MethylSeg Plotting Gallery

This notebook uses the bundled WGBS reference sample to demonstrate the main plotting controls. Each figure returns a Plotly or Matplotlib figure, so the same arguments work in scripts and notebooks.

In [ ]:
from pathlib import Path

import pandas as pd

from methylseg import MethylDataPrep, MethylSegPathway
from methylseg.helper_classes import DATA_DIR
from methylseg.utils import build_region_overlay_df, plot_interactive_beta_scatter

In [ ]:
REFERENCE_DIR = DATA_DIR / "reference_files"
OUT_DIR = "out" / "plotting_example_output"

PLOT_CHROM = "chr1"
WINDOW_START = 2_000_000
WINDOW_END = 4_000_000

# Reuse this mapping across every plot to keep biological states visually consistent.
STATE_COLORS = {
    "LOW": "#2166AC",
    "PMD": "#B2182B",
    "INTERMEDIATE": "#F4A261",
    "HIGH": "#1B9E77",
}

pd.Series(STATE_COLORS, name="hex color").rename_axis("state")

## Load the bundled model and sample

The saved WGBS model keeps this gallery fast enough to rerun while the sample preparation step exposes removed low-coverage-like CpGs for the genomic plots.

In [ ]:
model = MethylSegPathway.get_pretrained_model(OUT_DIR, resolution="wgbs")

sample_info, sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "WGBS_colon-primary-tumor_1_wgbs.tsv.gz",
    sample_id="colon-primary-tumor_1",
    resolution="wgbs",
    min_coverage=10,
    remove_low_coverage_like_cpgs=True,
).prepare()

regions_chr1 = model.generate_regions(sample_info=sample_info, chrom=PLOT_CHROM)
sample_info.sample_id, regions_chr1.shape

## Interpreting KMeans states

Feature distributions explain what each learned KMeans biological state represents in emission space. The histogram fills use the same `STATE_COLORS` mapping as the genomic and embedding plots, so the meaning of each color remains consistent throughout the gallery.

In [ ]:
model.assigner.plot_feature_distributions_by_kmeans_state(
    state_colors=STATE_COLORS
)

## Comparing KMeans and rule-based assignments

This confusion-matrix view compares the learned KMeans labels with rule-based assignments for the prepared sample on the selected chromosome.

In [ ]:
model.analyzer.evaluate_clustering_concordance(
    sample_info=sample_info, chrom=PLOT_CHROM
)

## Genomic label plots

Use `model.assigner.plot_labels(...)` when working directly with learned KMeans states. `model.analyzer.plot_labels(...)` can inspect KMeans and rule-based states with identical plot controls, while `model.plot_labels(label_source="hmm", ...)` renders HMM states. Pass `state_colors` to use a project palette, `sample_info_removed` to show filtered probes, and `max_points` to bound browser rendering for dense samples. `region_start` and `region_end` zoom the displayed genomic window without changing the segmentation.

In [ ]:
fig = model.assigner.plot_labels(
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom=PLOT_CHROM,
    label_title="KMeans biological state",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

In [ ]:
fig_kmeans = model.analyzer.plot_labels(
    label_source="kmeans",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    label_title="KMeans state for rule comparison",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

fig_rule_based = model.analyzer.plot_labels(
    label_source="rule_based",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    label_title="Rule-based state in a genomic window",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom=PLOT_CHROM,
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    label_title="HMM state in a genomic window",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

## Interval overlays

Cleaned-region overlays load previously generated PMD calls. A custom interval table uses its `state` column for state-colored overlays. The final example uses the lower-level scatter helper because it exposes `overlay_style="highlight"`, which colors a selected interval versus the rest of the chromosome.

In [ ]:
clean_summary_paths, clean_dir = model.get_clean_regions(
    regions_df=regions_chr1,
    sample_id=sample_info.sample_id,
    chrom=PLOT_CHROM,
)
clean_summary_paths, clean_dir

In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    use_cleaned_regions=True,
    overlay_state="PMD",
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    label_title="HMM state with cleaned PMD overlay",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

In [ ]:
# Any interval table with CpG_chrm, start, end, and state columns can recolor probes.
custom_overlay = regions_chr1.loc[:, ["CpG_chrm", "start", "end", "state"]].head(12).copy()
custom_overlay["state"] = custom_overlay["state"].map(
    lambda state: state.name if hasattr(state, "name") else str(state)
)
custom_overlay

In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    overlay_regions_df=custom_overlay,
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    label_title="HMM state with custom state overlay",
    max_points=60_000,
    state_colors=STATE_COLORS,
)

In [ ]:
highlight_overlay = build_region_overlay_df(
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    region_chrom=PLOT_CHROM,
    label="Selected window",
)
hmm_data, _ = model.segmentor.segment_sample(
    sample_info=sample_info, chrom=PLOT_CHROM
)

fig = plot_interactive_beta_scatter(
    df_plot=hmm_data,
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom=PLOT_CHROM,
    out_dir=str(OUT_DIR),
    label_col="hmm_state_readable",
    label_title="HMM state with highlighted window",
    overlay_regions_df=highlight_overlay,
    overlay_style="highlight",
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    max_points=60_000,
    state_colors=STATE_COLORS,
)

## Embedding plots

For a small number of points, use a scatter plot to inspect individual CpGs and PCA loading arrows. For dense data, use a hexbin plot and tune its grid size, minimum count, transparency, and border width. The genomic window arguments highlight overlapping CpGs in embedding space.

In [ ]:
fig = model.plot_embedding(
    label_source="kmeans",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    method="pca",
    hexbin=False,
    include_biplot=True,
    include_metrics=True,
    label_title="KMeans state",
    region_start=WINDOW_START,
    region_end=WINDOW_END,
    region_chrom=PLOT_CHROM,
    state_colors=STATE_COLORS,
)

In [ ]:
fig = model.plot_embedding(
    label_source="hmm",
    sample_info=sample_info,
    chrom=PLOT_CHROM,
    method="pca",
    hexbin=True,
    hexbin_gridsize=80,
    hexbin_mincnt=25,
    hexbin_alpha=0.85,
    hexbin_linewidths=0.15,
    include_biplot=False,
    include_metrics=False,
    label_title="HMM state density",
    state_colors=STATE_COLORS,
)